In [ ]:
#@title IFRS9 Modelo Matriz 2026 (USD + PEN) — 2026 trains on 2022–2024 + 2025 Jan–Aug (YTD)
from __future__ import annotations
import re, warnings
from typing import Dict, List, Optional, Tuple
from pathlib import Path
import numpy as np
import pandas as pd

# =========================
# ========== CONFIG =======
# =========================
# --- USD inputs ---
USD_MATRIX_FILE    = "Matriz de Riesgos 22-2025 USD.xlsx"
USD_EAD_FILE       = "EAD USD.xlsx"
USD_EAD_SHEET_NAME = "EAD USD"
USD_EAD_SCALE      = 1.0

# --- PEN inputs ---
PEN_MATRIX_FILE    = "Matriz de Riesgos 22-2025 PEN.xlsx"
PEN_EAD_FILE       = "EAD PEN.xlsx"
PEN_EAD_SHEET_NAME = "EAD PEN"
PEN_EAD_SCALE      = 1.0

"""
# --- FX for combining (PEN→USD) used to build 2025 currency shares ---
FX_PEN_USD_2025_CONST = 3.75
FX_PEN_USD_2025_CSV = None  # e.g., "fx_2025_pen_usd.csv"
"""

# --- Macros file used by both currencies ---
DATA_AJP_FILE = "Data AJP.xlsx"
MACRO_SHEET   = "modelos consolidado"

# Scenario macro inputs (Inflación, PBI, TasaRef, PBI Comercio, PBI Consumo Privado) → for 2025
baseline_input_2025   = np.array([0.025, 0.030, 0.0464, 0.030, 0.035], dtype=float)
optimistic_input_2025 = np.array([
    baseline_input_2025[0]*0.5,
    baseline_input_2025[1]*1.5,
    baseline_input_2025[2]*0.5,
    baseline_input_2025[3]*1.5,
    baseline_input_2025[4]*1.5
], dtype=float)
adverse_input_2025    = np.array([
    baseline_input_2025[0]*1.5,
    baseline_input_2025[1]*0.5,
    baseline_input_2025[2]*1.5,
    baseline_input_2025[3]*0.5,
    baseline_input_2025[4]*0.5
], dtype=float)


# ----- OPTIONAL: distinct 2026 macro scenario inputs (set arrays to override; else fallback to 2025)
baseline_input_2026   = np.array([0.020, 0.029, 0.0425, 0.028, 0.029], dtype=float)
#baseline 2026 were: 0.025, 0.032, 0.041, 0.030, 0.060
optimistic_input_2026 = np.array([
    baseline_input_2026[0]*0.5,
    baseline_input_2026[1]*1.5,
    baseline_input_2026[2]*0.5,
    baseline_input_2026[3]*1.5,
    baseline_input_2026[4]*1.5
], dtype=float)
adverse_input_2026    = np.array([
    baseline_input_2026[0]*1.5,
    baseline_input_2026[1]*0.5,
    baseline_input_2026[2]*1.5,
    baseline_input_2026[3]*0.5,
    baseline_input_2026[4]*0.5
], dtype=float)

# Year weights (models & seasonality)
YEAR_WEIGHTS_FOR = {
    2025: {2022: 0.10, 2023: 0.70, 2024: 1.00},
    2026: {2022: 0.20, 2023: 0.70, 2024: 1.00, 2025: 1.00},  # 2026 training can include 2025 YTD
}
YEAR_WEIGHTS = YEAR_WEIGHTS_FOR[2026]


#2023 used to have weight 0.7

AUTO_RELABEL_SCENARIOS = True
SEASON_ALPHA = 0.45      # declared but unused unless you apply the shrink ideas I shared earlier
POOL_EDGES   = True

FEATURES_MAIN  = ["Inflación","PBI","Tasa de Interés Referencial","PBI Comercio","PBI Consumo Privado"]
FEATURES_INTER = ["PBI*Inflacion","PBI*Consumo","TasaInt*Inflacion"]
FEATURES_ALL   = FEATURES_MAIN + FEATURES_INTER

TRAIN_START = pd.Timestamp("2022-01-01")
TRAIN_END   = pd.Timestamp("2025-12-31")

# include 'vigente'
IGNORE_BUCKETS: set[str] = set()

# =========================
# Canonicalization & helpers
# =========================
MANUAL_BUCKET_MAP: Dict[str, str] = {
    "vigente":"vigente","corriente":"vigente","current":"vigente",
    "r:1 - 30":"r:1-30","1-30 dias":"r:1-30",
    "31-60 dias":"r:31-60","61-90 dias":"r:61-90",
    "91-120 dias":"r:91-120","121-150 dias":"r:121-150",
    "151-180 dias":"r:151-180","181-360 dias":"r:181-360",
    "361+":"r:360+","360+":"r:360+","r:361+":"r:360+",
}
def _strip_accents(s: str) -> str:
    import unicodedata as _u
    return ''.join(c for c in _u.normalize('NFD', s) if _u.category(c) != 'Mn')

def canonicalize_bucket(name: str) -> str:
    if name is None or (isinstance(name, float) and np.isnan(name)): return ""
    s = str(name).strip()
    if s.lower() in MANUAL_BUCKET_MAP: return MANUAL_BUCKET_MAP[s.lower()]
    s = _strip_accents(s).lower()
    s = re.sub(r'\s+', ' ', s)
    s = s.replace(' :', ':').replace(': ', ':').replace(' - ', '-').replace(' – ', '-').replace(' — ', '-')
    s = s.replace(' +', '+').replace('+ ', '+')
    if re.search(r'\b(vigente|corriente|current)\b', s): return 'vigente'
    if re.search(r'\b(36[0-9]\+|360\+|361\+|r:\s*36[0-9]\+)\b', s): return 'r:360+'
    m = re.search(r'(r:\s*)?(\d+)\s*-\s*(\d+)', s)
    if m:
        lo, hi = int(m.group(2)), int(m.group(3))
        if lo == 0 and hi == 30: lo = 1
        return f"r:{lo}-{hi}"
    m2 = re.search(r'(\d+)\s*\+$', s)
    if m2 and int(m2.group(1)) >= 360: return 'r:360+'
    if s.startswith('r:') and re.search(r'r:\d+\-\d+', s): return s
    return s

def _extract_date(text: str) -> pd.Timestamp | pd.NaT:
    if pd.isna(text): return pd.NaT
    m = re.search(r"(\d{2})[/-](\d{2})[/-](\d{2,4})", str(text))
    if not m: return pd.NaT
    dd, mm, yy = m.groups(); yy = "20"+yy if len(yy)==2 else yy
    try: return pd.Timestamp(int(yy), int(mm), int(dd))
    except: return pd.NaT

def _dpd_from_bucket(bucket: str) -> float:
    b = str(bucket).strip().lower()
    if "vigente" in b or "corriente" in b or "current" in b: return 0.0
    if "360+" in b: return float("inf")
    m = re.search(r"(\d+)\s*-\s*(\d+)", b) or re.search(r"r:(\d+)-(\d+)", b)
    if m: return float(m.group(2))
    m = re.search(r"(\d+)", b); return float(m.group(1)) if m else 0.0

def _is_bucket_label(name: str) -> bool:
    s = canonicalize_bucket(name)
    if s in ("","nan"): return False
    return s=="vigente" or s=="r:360+" or bool(re.match(r"^r:\d+\-\d+$", s))

def _select_bucket_cols(cols: List[str], ignore: set[str]) -> List[str]:
    return [c for c in cols if _is_bucket_label(c) and canonicalize_bucket(c) not in ignore]

def _to_numeric_series(x: pd.Series) -> pd.Series:
    s = x.copy()
    if s.dtype == object:
        s = s.astype(str).str.replace("%","",regex=False)
        if s.str.contains(",", regex=False).mean()>0: s=s.str.replace(",",".",regex=False)
        s = s.str.replace(r"[^\d\.\-]", "", regex=True)
    s = pd.to_numeric(s, errors="coerce")
    if s.dropna().mean() > 1.5: s = s/100.0
    return s

def _parse_fecha_to_dt(s: str) -> pd.Timestamp:
    m = re.search(r"(\d{2})-(\d{2})-(\d{2,4})", str(s))
    if not m: return pd.NaT
    dd, mm, yy = m.groups(); yy = "20"+yy if len(yy)==2 else yy
    return pd.Timestamp(int(yy), int(mm), int(dd))

def _pick_date_series(df: pd.DataFrame) -> pd.Series:
    for cand in df.columns:
        if str(cand).strip().lower() in {"fecha","date"}:
            return pd.to_datetime(df[cand].apply(_parse_fecha_to_dt), errors="coerce")
    for cand in df.columns:
        if _is_bucket_label(cand): continue
        s = df[cand].astype(str)
        if s.str.contains(r"\b\d{2}-\d{2}-\d{2,4}\b", na=False).mean() >= 0.3:
            return pd.to_datetime(s.apply(_parse_fecha_to_dt), errors="coerce")
    if isinstance(df.index, (pd.DatetimeIndex, pd.PeriodIndex)):
        return pd.to_datetime(df.index.to_timestamp() if isinstance(df.index, pd.PeriodIndex) else df.index)
    return pd.to_datetime(pd.Series([pd.NaT]*len(df), index=df.index))

def _matrix_to_period(df_rates: pd.DataFrame) -> pd.DataFrame:
    df = df_rates.copy()
    df["__date__"] = _pick_date_series(df)
    df = df.dropna(subset=["__date__"]).set_index("__date__")
    df = df.rename(columns=lambda c: canonicalize_bucket(c))
    bucket_cols = _select_bucket_cols([c for c in df.columns if c != "Fecha"], set())
    return df[bucket_cols].astype(float).groupby(df.index.to_period("M")).mean()

# =========================
# Loaders
# =========================
def load_matrix_table(path: str) -> pd.DataFrame:
    sheets = pd.read_excel(path, sheet_name=None)
    best_df, best_name, best_score = None, None, -1
    for sh, raw in sheets.items():
        if not isinstance(raw, pd.DataFrame) or raw.empty: continue
        df = raw.copy()
        df = df.loc[~df.isna().all(axis=1)]
        if df.empty: continue
        df = df.loc[:, ~df.isna().all(axis=0)]
        if df.shape[1] < 2: continue
        first_col = df.columns[0]
        has_dates = df[first_col].astype(str).str.contains(r"\d{2}[-/]\d{2}[-/]\d{2,4}", na=False).any()
        bucket_like = sum(
            str(c).lower().startswith(("vigente","r:","corriente","current")) or
            bool(re.search(r"\d{1,3}\s*-\s*\d{1,3}", str(c).lower())) or "360+" in str(c)
            for c in df.columns[1:])
        score = int(has_dates) + bucket_like
        if score > best_score: best_df, best_name, best_score = df, sh, score
    if best_df is None: raise ValueError("No se encontró hoja tipo matriz.")
    print(f"[Matrix] Hoja: {best_name}  shape={best_df.shape}")
    return best_df

def load_ead_table(path: str, forced_sheet: Optional[str], scale: float=1.0) -> pd.DataFrame:
    if not Path(path).exists(): raise FileNotFoundError(f"EAD file not found: {path}")
    sheets = pd.read_excel(path, sheet_name=None)
    if forced_sheet not in sheets: raise ValueError(f"EAD sheet '{forced_sheet}' not found. Available: {list(sheets.keys())}")
    df = sheets[forced_sheet].copy()
    print(f"[EAD] Hoja forzada: {forced_sheet}  shape={df.shape}")

    cols = list(df.columns); first_col = cols[0]

    def _looks_like_date_str(x: str) -> bool:
        if pd.isna(x): return False
        s = str(x).strip()
        try: pd.to_datetime(s, errors="raise"); return True
        except: return bool(re.search(r"\b\d{2}[/-]\d{2}[/-]\d{2,4}\b", s))

    def _parse_date(s):
        try: return pd.to_datetime(s)
        except:
            m = re.search(r"(\d{2})[/-](\d{2})[/-](\d{2,4})", str(s))
            if not m: return pd.NaT
            dd, mm, yy = m.groups(); yy = "20"+yy if len(yy)==2 else yy
            try: return pd.Timestamp(int(yy), int(mm), int(dd))
            except: return pd.NaT

    def _looks_like_bucket_label(x: str) -> bool:
        if pd.isna(x): return False
        s = canonicalize_bucket(str(x))
        if s in ("","nan"): return False
        return s=="vigente" or "360+" in s or bool(re.search(r"r:\d+\-\d+", s)) or bool(re.search(r"\b\d+\s*-\s*\d+\b", s))

    n_cols_dates = sum(_looks_like_date_str(c) for c in cols)
    n_cols_buckets = sum(_looks_like_bucket_label(c) for c in cols)
    first_vals = df[first_col].dropna().astype(str).head(20).tolist()
    n_firstcol_dates = sum(_looks_like_date_str(v) for v in first_vals)
    n_firstcol_buckets = sum(_looks_like_bucket_label(v) for v in first_vals)

    def build_ts_rows_buckets():
        df_idx = df[first_col].astype(str).map(canonicalize_bucket)
        df2 = df.drop(columns=[first_col])
        parsed_cols = [_parse_date(c) for c in df2.columns]
        keep = [i for i,d in enumerate(parsed_cols) if pd.notna(d)]
        df2 = df2.iloc[:, keep]
        parsed_cols = [parsed_cols[i] for i in keep]
        df2 = df2.apply(lambda s: pd.to_numeric(s, errors="coerce"))
        ts = df2.T
        ts.index = pd.to_datetime(parsed_cols)
        ts.columns = df_idx.values
        ts = ts.loc[:, ts.notna().any(axis=0)].astype(float).sort_index()
        ts.columns = [canonicalize_bucket(c) for c in ts.columns]
        return ts

    def build_ts_rows_dates():
        df2 = df.copy(); df2["__date__"] = df2[first_col].map(_parse_date)
        df2 = df2[~df2["__date__"].isna()].sort_values("__date__")
        bucket_cols = [c for c in df2.columns if c not in (first_col, "__date__")]
        canon_map = {c: canonicalize_bucket(c) for c in bucket_cols}
        df2 = df2.rename(columns=canon_map)
        for c in canon_map.values(): df2[c] = pd.to_numeric(df2[c], errors="coerce")
        ts = df2.set_index("__date__")[sorted(set(canon_map.values()))].astype(float)
        ts = ts.loc[:, ts.notna().any(axis=0)]
        return ts

    if n_cols_dates >= max(6, n_cols_buckets) and n_firstcol_buckets >= n_firstcol_dates:
        ts = build_ts_rows_buckets(); print("[EAD] Orient: filas=buckets, columnas=fechas (transpuesto).")
    elif n_firstcol_dates >= max(6, n_firstcol_buckets):
        ts = build_ts_rows_dates(); print("[EAD] Orient: filas=fechas, columnas=buckets.")
    else:
        ts = build_ts_rows_buckets(); print("[EAD] Orient: fallback.")

    ts = ts * float(scale)
    return ts

# =========================
# Macros
# =========================
def load_macros():
    df_macro_port = pd.read_excel(DATA_AJP_FILE, sheet_name=MACRO_SHEET, usecols="A,B,C,D,E,F,G,H,I").copy()
    year_col = next((c for c in df_macro_port.columns if str(c).strip().lower() in {"año","ano","year","periodo","period"}), None)
    df_m = df_macro_port[[*([year_col] if year_col else []), *FEATURES_MAIN]].dropna(how="any").copy()
    if year_col: df_m["Year"] = pd.to_numeric(df_m[year_col], errors="coerce")
    else:
        yrs = [2022, 2023, 2024, 2025]
        if len(df_m) >= len(yrs): df_m = df_m.tail(len(yrs)).copy(); df_m["Year"] = yrs
        else: raise ValueError("Agrega Año/Year en Data AJP.xlsx.")
    macro_y = (df_m.dropna().astype({"Year":int}).drop_duplicates(subset=["Year"])
                 .set_index("Year").sort_index().reindex([2022,2023,2024,2025]))
    macro_y["PBI*Inflacion"]     = macro_y["PBI"] * macro_y["Inflación"]
    macro_y["PBI*Consumo"]       = macro_y["PBI"] * macro_y["PBI Consumo Privado"]
    macro_y["TasaInt*Inflacion"] = macro_y["Tasa de Interés Referencial"] * macro_y["Inflación"]
    return macro_y

# =========================
# Modeling utils
# =========================
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, BayesianRidge
EPS = 1e-5
def logit(p): p = np.clip(p, EPS, 1-EPS); return np.log(p) - np.log(1-p)
def expit(z): return 1.0 / (1.0 + np.exp(-z))

def fit_small_models_logit(X: pd.DataFrame, y_rate: pd.Series, sample_weight: Optional[np.ndarray] = None):
    z = logit(y_rate.values.astype(float))
    ridge = make_pipeline(StandardScaler(with_mean=True), RidgeCV(alphas=[1e-3,1e-2,1e-1,1,10,100]))
    br    = make_pipeline(StandardScaler(with_mean=True), BayesianRidge())
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if sample_weight is None:
            ridge.fit(X.values, z); br.fit(X.values, z)
        else:
            try:
                ridge.fit(X.values, z, standardscaler__sample_weight=sample_weight, ridgecv__sample_weight=sample_weight)
            except TypeError:
                ridge.fit(X.values, z, ridgecv__sample_weight=sample_weight)
            try:
                br.fit(X.values, z, standardscaler__sample_weight=sample_weight, bayesianridge__sample_weight=sample_weight)
            except TypeError:
                br.fit(X.values, z, bayesianridge__sample_weight=sample_weight)
    return ridge, br

def predict_rate_from_logit_models(ridge, br, xrow: np.ndarray) -> float:
    zr = float(ridge.predict(xrow.reshape(1,-1))[0])
    zb = float(br.predict(xrow.reshape(1,-1))[0])
    return float(expit((zr + zb) / 2.0))

# =========================
# Currency run
# =========================
def build_currency_run(tag: str, MATRIX_FILE: str, EAD_FILE: str, EAD_SHEET_NAME: str, EAD_SCALE: float,
                       macro_y_in: pd.DataFrame):
    print(f"\n==== Processing currency: {tag} ====")

    # Matrix
    matrix_raw = load_matrix_table(MATRIX_FILE)
    date_col = matrix_raw.columns[0]
    df_mat = matrix_raw.copy()
    df_mat["__date__"] = df_mat[date_col].apply(_extract_date)
    df_mat = df_mat[~df_mat["__date__"]].isna() if "__date__" not in df_mat.columns else df_mat
    df_mat = df_mat[~df_mat["__date__"].isna()].sort_values("__date__")
    df_mat = df_mat[(df_mat["__date__"] >= TRAIN_START) & (df_mat["__date__"] <= TRAIN_END)].copy()
    for c in [c for c in df_mat.columns if c not in (date_col,"__date__")]: df_mat[c] = _to_numeric_series(df_mat[c])
    ts_monthly_all = df_mat.set_index("__date__").dropna(how="all")
    ts_monthly_all = ts_monthly_all.rename(columns=canonicalize_bucket)

    # Canonical, global bucket order (incl. vigente)
    raw_buckets = [c for c in ts_monthly_all.columns if _is_bucket_label(c) and canonicalize_bucket(c) not in IGNORE_BUCKETS]
    BUCKET_ORDER = sorted(pd.Index(raw_buckets).unique().tolist(), key=_dpd_from_bucket)
    ts_monthly_all = ts_monthly_all.reindex(columns=BUCKET_ORDER)
    print(f"[{tag}] Bucket order: {BUCKET_ORDER}")

    annual_all = ts_monthly_all.resample("YE").mean(); annual_all.index = annual_all.index.year

    # Macros & training bases
    macro_y = macro_y_in.copy()
    base_train_years = [y for y in [2022, 2023, 2024] if y in annual_all.index and y in macro_y.index]
    if len(base_train_years) < 3: raise ValueError(f"[{tag}] Need 2022–2024; found {base_train_years}")
    annual_base  = annual_all.loc[base_train_years].reindex(columns=BUCKET_ORDER).copy()
    macro_y_2225 = macro_y.loc[[*base_train_years, 2025]].copy()
    ts_monthly = ts_monthly_all.reindex(columns=BUCKET_ORDER)

    # ===== Seasonality (weighted, pooled, renorm) — still 2022–2024 only =====
    idx_by_year = {}
    for yr, df_y in ts_monthly.groupby(ts_monthly.index.year):
        if yr not in base_train_years: continue
        monthly = df_y.groupby(df_y.index.month).mean()
        year_mean = df_y.mean()
        idx_by_year[yr] = monthly.divide(year_mean, axis=1)
    all_months = sorted(set().union(*[df.index for df in idx_by_year.values()]))
    seasonal_idx = pd.DataFrame(index=all_months, columns=BUCKET_ORDER, dtype=float)
    for m in all_months:
        for b in BUCKET_ORDER:
            num = den = 0.0
            for yr, df in idx_by_year.items():
                val = df.loc[m, b] if (m in df.index and b in df.columns) else np.nan
                if pd.notna(val):
                    w = YEAR_WEIGHTS.get(yr, 1.0)
                    num += w * float(val); den += w
            seasonal_idx.loc[m, b] = (num / den) if den > 0 else 1.0
    seasonal_idx = seasonal_idx.fillna(1.0).apply(lambda c: c / (c.mean() if pd.notna(c.mean()) and c.mean()!=0 else 1.0), axis=0)
    if POOL_EDGES:
        if 1 in seasonal_idx.index and 2 in seasonal_idx.index:  seasonal_idx.loc[1]  = 0.5*seasonal_idx.loc[1]  + 0.5*seasonal_idx.loc[2]
        if 12 in seasonal_idx.index and 11 in seasonal_idx.index: seasonal_idx.loc[12] = 0.5*seasonal_idx.loc[12] + 0.5*seasonal_idx.loc[11]
        seasonal_idx = seasonal_idx.apply(lambda c: c / (c.mean() if pd.notna(c.mean()) and c.mean()!=0 else 1.0), axis=0)

    floors = ts_monthly.quantile(0.05, numeric_only=True).fillna(0.0)
    caps   = ts_monthly.quantile(0.95, numeric_only=True).clip(upper=1.0).fillna(1.0)
    for b in BUCKET_ORDER:
        if _dpd_from_bucket(b) >= 360 or "360+" in str(b): floors[b] = 1.0; caps[b] = 1.0

    # ==== 2025 YTD (Jan–Aug) add-on for 2026 training ====
    # Build a per-bucket YTD mean for 2025 (Jan..Aug). If not available, we fall back to 22–24 only.
    mat_2025_ytd = ts_monthly_all[(ts_monthly_all.index.year == 2025) & (ts_monthly_all.index.month <= 8)]
    have_2025_ytd = not mat_2025_ytd.empty
    if have_2025_ytd:
        # monthly mean per bucket across Jan..Aug
        annual_2025_ytd_series = mat_2025_ytd.resample("M").mean().mean(axis=0).reindex(BUCKET_ORDER)
        print(f"[{tag}] 2025 YTD (Jan–Aug) available — will be added to 2026 training.")
    else:
        annual_2025_ytd_series = None
        print(f"[{tag}] 2025 YTD (Jan–Aug) NOT found — 2026 training will use 2022–2024 only.")

    # ===== 4) Models (two fits):
    #   A) for 2025 predictions (train on 2022–2024)
    #   B) for 2026 predictions (train on 2022–2024 + 2025YTD if available)
    def _mk_scenario_rows(inp):
        return pd.Series(np.append(inp, [inp[1]*inp[0], inp[1]*inp[4], inp[2]*inp[0]]),
                         index=FEATURES_ALL, dtype=float)

    SCENARIO_ROWS_2025 = {
        "Optimistic": _mk_scenario_rows(optimistic_input_2025),
        "Baseline":   _mk_scenario_rows(baseline_input_2025),
        "Adverse":    _mk_scenario_rows(adverse_input_2025),
    }
    SCENARIO_ROWS_2026 = {
        "Optimistic": _mk_scenario_rows(optimistic_input_2026 if isinstance(optimistic_input_2026, np.ndarray) else optimistic_input_2025),
        "Baseline":   _mk_scenario_rows(baseline_input_2026   if isinstance(baseline_input_2026,   np.ndarray) else baseline_input_2025),
        "Adverse":    _mk_scenario_rows(adverse_input_2026    if isinstance(adverse_input_2026,    np.ndarray) else adverse_input_2025),
    }

    pred_mats_annual_2025: Dict[str, pd.Series] = {sc: pd.Series(index=BUCKET_ORDER, dtype=float) for sc in SCENARIO_ROWS_2025}
    pred_mats_annual_2026: Dict[str, pd.Series] = {sc: pd.Series(index=BUCKET_ORDER, dtype=float) for sc in SCENARIO_ROWS_2026}

    X_base = macro_y_2225.loc[base_train_years, FEATURES_ALL].astype(float).copy()
    sw_base = np.array([YEAR_WEIGHTS.get(y, 1.0) for y in base_train_years], dtype=float)

    if have_2025_ytd:
        train_years_26 = [*base_train_years, 2025]
        X_26 = macro_y_2225.loc[train_years_26, FEATURES_ALL].astype(float).copy()
        sw_26 = np.array([YEAR_WEIGHTS.get(y, 1.0) for y in train_years_26], dtype=float)
    else:
        train_years_26 = base_train_years[:]
        X_26 = X_base.copy()
        sw_26 = sw_base.copy()

    for bucket in BUCKET_ORDER:
        # y for 2025 predictions: 2022–2024 only
        yb_base = annual_base[bucket].astype(float)

        # y for 2026 predictions: 2022–2024 + 2025YTD (if exists)
        if have_2025_ytd and pd.notna(annual_2025_ytd_series.get(bucket, np.nan)):
            yb_26 = pd.concat([annual_base[bucket].astype(float), pd.Series({2025: float(annual_2025_ytd_series[bucket])})])
        else:
            yb_26 = annual_base[bucket].astype(float)

        try:
            # Fit models for 2025 (base) and 2026 (with YTD)
            ridge_b_base, br_b_base = fit_small_models_logit(X_base, yb_base, sample_weight=sw_base)
            ridge_b_26,  br_b_26  = fit_small_models_logit(X_26,  yb_26,  sample_weight=sw_26)

            # Predict annual by scenario
            for scen, xser in SCENARIO_ROWS_2025.items():
                pred_mats_annual_2025[scen][bucket] = predict_rate_from_logit_models(ridge_b_base, br_b_base, xser.values.astype(float))
            for scen, xser in SCENARIO_ROWS_2026.items():
                pred_mats_annual_2026[scen][bucket] = predict_rate_from_logit_models(ridge_b_26, br_b_26, xser.values.astype(float))

        except Exception as e:
            last_val_base = float(yb_base.iloc[-1])
            last_val_26   = float(yb_26.iloc[-1])
            for scen in pred_mats_annual_2025: pred_mats_annual_2025[scen][bucket] = last_val_base
            for scen in pred_mats_annual_2026: pred_mats_annual_2026[scen][bucket] = last_val_26
            print(f"[{tag}] Fallback used for bucket '{bucket}': {e}")

    def _post_annual(s: pd.Series) -> pd.Series:
        s = s.astype(float).clip(lower=floors.reindex(s.index).fillna(0.0),
                                 upper=caps.reindex(s.index).fillna(1.0))
        return s.reindex(BUCKET_ORDER).loc[BUCKET_ORDER].cummax()

    for dct in (pred_mats_annual_2025, pred_mats_annual_2026):
        for scen in dct:
            dct[scen] = _post_annual(dct[scen])

    # ===== 5) Relabel by 2025 EAD-weighted severity (incl. vigente) =====
    ead_ts = load_ead_table(EAD_FILE, forced_sheet=EAD_SHEET_NAME, scale=EAD_SCALE)
    ead_monthly = ead_ts.resample("M").sum(numeric_only=True); ead_monthly.index = ead_monthly.index.to_period("M")
    ead_monthly = ead_monthly.rename(columns=lambda c: canonicalize_bucket(c)).reindex(columns=BUCKET_ORDER)

    def _ead_weights_from_ead_monthly_2025(ead_m: Optional[pd.DataFrame], buckets: List[str]) -> Optional[pd.Series]:
        if ead_m is None or ead_m.empty: return None
        e25 = ead_m[ead_m.index.year == 2025].copy()
        if e25.empty: return None
        w = e25.reindex(columns=buckets, fill_value=0.0).sum(axis=0).astype(float)
        return w if float(w.sum())>0 else None

    def _relabel(preds: Dict[str, pd.Series], ead_m: pd.DataFrame, buckets: List[str], eps: float = 5e-5) -> Dict[str, pd.Series]:
        w = _ead_weights_from_ead_monthly_2025(ead_m, buckets)
        if w is None: return preds
        scores = {}
        unweighted = {sc: float(preds[sc].mean()) for sc in preds}
        for sc in preds:
            s = preds[sc].reindex(w.index).fillna(0.0)
            sev = float((s*w).sum()/(float(w.sum())+1e-8))
            scores[sc] = (sev, unweighted[sc])
        ordered = sorted(scores.items(), key=lambda kv: (round(kv[1][0]/eps)*eps, kv[1][1]))
        names = [sc for sc,_ in ordered]
        mapping = {"Optimistic": names[0], "Baseline": names[1], "Adverse": names[2]}
        return {std: preds[mapping[std]] for std in ["Optimistic","Baseline","Adverse"]}

    if AUTO_RELABEL_SCENARIOS:
        pred_mats_annual_2025 = _relabel(pred_mats_annual_2025, ead_monthly, BUCKET_ORDER)
        pred_mats_annual_2026 = _relabel(pred_mats_annual_2026, ead_monthly, BUCKET_ORDER)

    # ===== 6) Calibration per year (identity slope/intercept by default) =====
    def _retune_to_target_mean(p_series, target_mean):
        EPS_ = 1e-6
        def _logit(p): p = np.clip(p, EPS_, 1-EPS_); return np.log(p) - np.log(1-p)
        def _expit(z): return 1.0 / (1.0 + np.exp(-z))
        idx = p_series.index
        z = pd.Series(_logit(p_series.values), index=idx, dtype=float)
        lo, hi = -5.0, 5.0
        for _ in range(50):
            mid = 0.5*(lo+hi)
            m = float(_expit(z + mid).mean())
            if m < target_mean: lo = mid
            else: hi = mid
        return pd.Series(_expit(z + 0.5*(lo+hi)), index=idx)

    def _calibrate_dict(dct: Dict[str, pd.Series]) -> Dict[str, pd.Series]:
        out = {}
        for scen, s in dct.items():
            target = float(s.mean())  # identity mean-preserving
            tuned = _retune_to_target_mean(s.astype(float), target).clip(0.0, 1.0)
            tuned = tuned.reindex(BUCKET_ORDER).loc[BUCKET_ORDER].cummax()
            out[scen] = tuned
        return out

    pred_mats_annual_2025 = _calibrate_dict(pred_mats_annual_2025)
    pred_mats_annual_2026 = _calibrate_dict(pred_mats_annual_2026)

    # ===== 7) Monthly matrices (distinct by year) =====
    def monthly_matrix_from_annual_for_year(s_annual: pd.Series, seasonal_idx_local: pd.DataFrame, year: int) -> pd.DataFrame:
        s_annual = s_annual.reindex(BUCKET_ORDER)
        rows = []
        for m in range(1,13):
            label = f"Matriz provisiones {pd.Timestamp(year, m, 1) + pd.offsets.MonthEnd(0):%d-%m-%y}"
            mult = seasonal_idx_local.loc[m, BUCKET_ORDER].astype(float) if m in seasonal_idx_local.index else pd.Series(1.0, index=BUCKET_ORDER)
            row_vals = (s_annual * mult).clip(0.0, 1.0).loc[BUCKET_ORDER].cummax()
            rows.append(pd.Series([label] + list(row_vals.values), index=["Fecha"] + BUCKET_ORDER))
        df = pd.DataFrame(rows)
        return df[["Fecha"] + BUCKET_ORDER]

    matrices_2025 = {sc: monthly_matrix_from_annual_for_year(pred_mats_annual_2025[sc], seasonal_idx, 2025) for sc in pred_mats_annual_2025}
    matrices_2026 = {sc: monthly_matrix_from_annual_for_year(pred_mats_annual_2026[sc], seasonal_idx, 2026) for sc in pred_mats_annual_2026}

    # ===== 8) Summaries & Errors =====
    def _summary_for_2025(matrices_2025: Dict[str, pd.DataFrame], ead_m: Optional[pd.DataFrame]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:  # noqa
        diags = {}; ead_2025_m = ead_m[ead_m.index.year == 2025].copy() if (ead_m is not None and not ead_m.empty) else None
        rows = []
        for scen, mat in matrices_2025.items():
            mm = mat.copy(); mm["__date__"] = _pick_date_series(mm)
            mm = mm.dropna(subset=["__date__"]).set_index("__date__")
            mm = mm.rename(columns=lambda c: canonicalize_bucket(c) if c != "Fecha" else c)
            bucket_cols = [b for b in BUCKET_ORDER if b in mm.columns]
            if not bucket_cols:
                rows.append({"Scenario":scen,"Mean_UnweightedRate_2025":np.nan,"Mean_EAD_WeightedRate_2025":np.nan,"EAD_Months_Used_2025":0,"Bucket_Count_Intersect":0})
                diags[scen] = pd.DataFrame([{"MonthPeriod":"FALLBACK_USED"}]); continue
            mat_m = mm[bucket_cols].astype(float).groupby(mm.index.to_period("M")).mean()
            unweighted_all_12m = float(mat_m.mean(axis=1).mean()) if not mat_m.empty else np.nan
            mean_unw = np.nan; mean_eadw = np.nan; months_used = 0; intersect = 0; drows=[]
            if ead_2025_m is not None and not ead_2025_m.empty and not mat_m.empty:
                common = [b for b in bucket_cols if b in ead_2025_m.columns]; intersect = len(common)
                if intersect>0:
                    M = mat_m[common].copy(); E = ead_2025_m.reindex(M.index).fillna(0.0).astype(float)
                    eligible = [p for p in M.index if float(E.loc[p, common].sum())>0]
                    months_used = len(eligible)
                    if months_used>0:
                        mean_unw = float(M.loc[eligible].mean(axis=1).mean())
                        ecl_sum=ead_sum=0.0
                        for p in eligible:
                            rates = M.loc[p, common].astype(float); eads = E.loc[p, common].astype(float)
                            ecl_sum += float((rates*eads).sum()); ead_sum += float(eads.sum())
                            drows.append({"MonthPeriod":str(p),"SumEAD_AllBuckets":float(eads.sum()),
                                          "UnweightedMean":float(rates.mean()),"ECL_contrib_All":float((rates*eads).sum()),
                                          "Denominator_All":float(eads.sum())})
                        mean_eadw = float(ecl_sum/(ead_sum+1e-8)) if ead_sum>0 else np.nan
            if np.isnan(mean_unw): mean_unw = unweighted_all_12m
            if np.isnan(mean_eadw): mean_eadw = unweighted_all_12m
            diags[scen] = pd.DataFrame(drows) if drows else pd.DataFrame([{"MonthPeriod":"FALLBACK_USED"}])
            rows.append({"Scenario":scen,"Mean_UnweightedRate_2025":mean_unw,"Mean_EAD_WeightedRate_2025":mean_eadw,
                         "EAD_Months_Used_2025":months_used,"Bucket_Count_Intersect":intersect})
        return pd.DataFrame(rows).set_index("Scenario"), diags

    def _errors_2025_for_scenario(scen_name: str,
                                  matrices_2025: Dict[str, pd.DataFrame],
                                  actual_monthly_all: pd.DataFrame,
                                  ead_monthly: Optional[pd.DataFrame]) -> dict:
        mm = matrices_2025[scen_name].copy()
        mm["__date__"] = _pick_date_series(mm)
        mm = mm.dropna(subset=["__date__"]).set_index("__date__")
        mm = mm.rename(columns=lambda c: canonicalize_bucket(c) if c != "Fecha" else c)
        pred_m = mm[[b for b in BUCKET_ORDER if b in mm.columns]] \
                   .astype(float).groupby(mm.index.to_period("M")).mean()

        act = actual_monthly_all.rename(columns=canonicalize_bucket)
        act = act[act.index.year == 2025].copy()
        act_m = act.groupby(act.index.to_period("M")).mean()

        common = [b for b in BUCKET_ORDER if (b in pred_m.columns and b in act_m.columns)]
        if not common:
            return {"Scenario": scen_name, "MAE_Unweighted_2025": np.nan, "MAE_EAD_Weighted_2025": np.nan,
                    "PortfolioAbsErr_Unweighted_2025": np.nan, "PortfolioAbsErr_EAD_Weighted_2025": np.nan,
                    "EligibleMonths": 0, "BucketCount": 0}

        P = pred_m[common]; A = act_m[common]
        months = sorted(list(set(P.index).intersection(set(A.index))))

        E = None
        if ead_monthly is not None and not ead_monthly.empty:
            E = ead_monthly.rename(columns=canonicalize_bucket).reindex(columns=common)
            E = E[E.index.year == 2025].copy()

        abs_err = (P.loc[months] - A.loc[months]).abs()
        mae_unw = float(abs_err.stack().mean()) if not abs_err.empty else np.nan

        mae_eadw = np.nan
        if E is not None and not E.empty:
            num = den = 0.0
            for p in months:
                if p in E.index:
                    e = E.loc[p].astype(float).fillna(0.0)
                    if float(e.sum()) > 0:
                        err = abs_err.loc[p].astype(float).fillna(0.0)
                        num += float((err * e).sum()); den += float(e.sum())
            if den > 0: mae_eadw = float(num / den)

        port_pred_unw = P.loc[months].mean(axis=1); port_act_unw = A.loc[months].mean(axis=1)
        port_abs_err_unw = float((port_pred_unw - port_act_unw).abs().mean())

        port_abs_err_eadw = np.nan
        if E is not None and not E.empty:
            num = den = 0.0
            for p in months:
                if p in E.index:
                    e = E.loc[p].astype(float).fillna(0.0); se = float(e.sum())
                    if se > 0:
                        rp = float((P.loc[p] * e).sum() / se)
                        ra = float((A.loc[p] * e).sum() / se)
                        num += se * abs(rp - ra); den += se
            if den > 0: port_abs_err_eadw = float(num / den)

        return {"Scenario": scen_name, "MAE_Unweighted_2025": mae_unw, "MAE_EAD_Weighted_2025": mae_eadw,
                "PortfolioAbsErr_Unweighted_2025": port_abs_err_unw,
                "PortfolioAbsErr_EAD_Weighted_2025": port_abs_err_eadw,
                "EligibleMonths": len(months), "BucketCount": len(common)}

    summary_2025, diag_2025 = _summary_for_2025(matrices_2025, ead_monthly)
    errors_rows = [_errors_2025_for_scenario(s, matrices_2025, ts_monthly_all, ead_monthly)
                   for s in ["Optimistic","Baseline","Adverse"]]
    errors_2025 = pd.DataFrame(errors_rows).set_index("Scenario")

    # Export ordered sheets per currency
    out_name = f"Provision_Matrices_2025_2026_{tag}.xlsx"
    with pd.ExcelWriter(out_name, engine="openpyxl") as xw:
        for scen, df_out in matrices_2025.items():
            df_out = df_out[["Fecha"] + BUCKET_ORDER]
            df_out.to_excel(xw, sheet_name=f"2025_{scen}", index=False)
        for scen, df_out in matrices_2026.items():
            df_out = df_out[["Fecha"] + BUCKET_ORDER]
            df_out.to_excel(xw, sheet_name=f"2026_{scen}", index=False)
        summary_2025.to_excel(xw, sheet_name="Annual_Summary_2025", index=True, float_format="%.6f")
        for scen, ddf in diag_2025.items():
            ddf.to_excel(xw, sheet_name=f"Diag2025_{scen}", index=False)
        errors_2025.to_excel(xw, sheet_name="Errors_2025", index=True, float_format="%.6f")
    print(f"[{tag}] Saved: {out_name}")

    return {"mat2025": matrices_2025, "mat2026": matrices_2026,
            "ead_monthly": ead_monthly, "bucket_order": BUCKET_ORDER}

# =========================
# Run both currencies
# =========================
def load_fx_2025_series(months_needed):
    if FX_PEN_USD_2025_CSV and Path(FX_PEN_USD_2025_CSV).exists():
        fx = pd.read_csv(FX_PEN_USD_2025_CSV)
        fx["Month"] = pd.PeriodIndex(pd.to_datetime(fx["Month"]+"-01"), freq="M")
        s = fx.set_index("Month")["FX"].astype(float)
        return s.reindex(months_needed).fillna(method="ffill").fillna(method="bfill")
    return pd.Series({m: float(FX_PEN_USD_2025_CONST) for m in months_needed})

macro_y_all = load_macros()
usd_res = build_currency_run("USD", USD_MATRIX_FILE, USD_EAD_FILE, USD_EAD_SHEET_NAME, USD_EAD_SCALE, macro_y_all)
pen_res = build_currency_run("PEN", PEN_MATRIX_FILE, PEN_EAD_FILE, PEN_EAD_SHEET_NAME, PEN_EAD_SCALE, macro_y_all)

##############################################################################
#Code For Printing 2026 Summary (unchanged)
#There is no 2026 EAD so weighted numbers might be unreliable
################################################################################
def annual_summary_2026_from_result(res: dict, tag: str) -> pd.DataFrame:
    """
    Builds a 2026 summary per scenario from res['mat2026'].
    EAD-weighted mean uses 2025 bucket mix as a proxy.
    Writes 'Annual_Summary_2026' and 'Diag2026_*' sheets into the currency workbook.
    """
    BUCKET_ORDER = res["bucket_order"]
    mats_2026 = res["mat2026"]                         # dict: scenario -> DataFrame with 'Fecha' + buckets
    ead_monthly = res.get("ead_monthly", None)         # monthly EAD by bucket (Period[M] index)

    # Build a fixed bucket weight vector from 2025 EAD (sum across 2025 months)
    w = None
    if ead_monthly is not None and not ead_monthly.empty:
        e25 = ead_monthly[ead_monthly.index.year == 2025].copy()
        if not e25.empty:
            w = (e25.reindex(columns=BUCKET_ORDER, fill_value=0.0)
                    .sum(axis=0).astype(float))
            if float(w.sum()) <= 0:
                w = None

    rows, diags = [], {}
    for scen, mat in mats_2026.items():
        mm = mat.copy()
        mm["__date__"] = _pick_date_series(mm)
        mm = mm.dropna(subset=["__date__"]).set_index("__date__")
        mm = mm.rename(columns=lambda c: canonicalize_bucket(c) if c != "Fecha" else c)

        bucket_cols = [b for b in BUCKET_ORDER if b in mm.columns]
        if not bucket_cols or mm.empty:
            rows.append({
                "Scenario": scen,
                "Mean_UnweightedRate_2026": np.nan,
                "Mean_EAD_WeightedRate_2026": np.nan,
                "EAD_Source_Year": None,
                "Months_Used": 0,
                "Bucket_Count": 0
            })
            diags[scen] = pd.DataFrame([{"MonthPeriod": "NO_DATA"}])
            continue

        # Monthly matrix for 2026
        mat_m = mm[bucket_cols].astype(float).groupby(mm.index.to_period("M")).mean()
        months_used = len(mat_m.index)

        # Unweighted: mean of monthly portfolio simple average
        mean_unw = float(mat_m.mean(axis=1).mean()) if months_used > 0 else np.nan

        # EAD-weighted using the 2025 bucket mix (constant across 2026 months)
        if w is not None and months_used > 0:
            w_use = w.reindex(bucket_cols).fillna(0.0).astype(float)
            ecl_sum = ead_sum = 0.0
            for p in mat_m.index:
                rates = mat_m.loc[p, bucket_cols].astype(float)
                ecl_sum += float((rates * w_use).sum())
                ead_sum += float(w_use.sum())  # same mix each month
            mean_eadw = float(ecl_sum / (ead_sum + 1e-8)) if ead_sum > 0 else np.nan
            ead_src_year = 2025
        else:
            mean_eadw = mean_unw
            ead_src_year = None

        rows.append({
            "Scenario": scen,
            "Mean_UnweightedRate_2026": mean_unw,
            "Mean_EAD_WeightedRate_2026": mean_eadw,
            "EAD_Source_Year": ead_src_year,
            "Months_Used": months_used,
            "Bucket_Count": len(bucket_cols)
        })

        # Minimal diagnostics: per-month unweighted portfolio mean
        diags[scen] = pd.DataFrame({
            "MonthPeriod": [str(p) for p in mat_m.index],
            "UnweightedMean": mat_m.mean(axis=1).astype(float).values
        })

    df_summary = pd.DataFrame(rows).set_index("Scenario")

    # Append to the per-currency workbook created earlier
    out_name = f"Provision_Matrices_2025_2026_{tag}.xlsx"
    with pd.ExcelWriter(out_name, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
        df_summary.to_excel(xw, sheet_name="Annual_Summary_2026", float_format="%.6f")
        for scen, ddf in diags.items():
            ddf.to_excel(xw, sheet_name=f"Diag2026_{scen}", index=False)
    print(f"[{tag}] Annual_Summary_2026 written to {out_name}")

    return df_summary

# ---- Run for both currencies (after your existing usd_res/pen_res lines) ----
usd_sum26 = annual_summary_2026_from_result(usd_res, "USD")
pen_sum26 = annual_summary_2026_from_result(pen_res, "PEN")

print("\n=== USD – Annual Summary 2026 ===")
print(usd_sum26)
print("\n=== PEN – Annual Summary 2026 ===")
print(pen_sum26)


KeyboardInterrupt: 